# Final Project

Author: Evan Whitfield

Course: ST 554

Purpose: Final Project

## Part 1 - Fitting Your Model

In [1]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

In [2]:
power_ml_data = pd.read_csv('power_ml_data.csv')

In [28]:
spark_sess = SparkSession.builder.getOrCreate()

spark_df = spark_sess.createDataFrame(power_ml_data)

In [29]:
spark_df.show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

In [30]:
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [31]:
cast_hour_double = SQLTransformer(statement = 
    """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double
    FROM __THIS__
    """
)

In [32]:
binary_day_night = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="day_night")

In [33]:
one_hot = OneHotEncoder(inputCols=["Month"],outputCols=["Month_ohe"],dropLast=False)

In [34]:
weather_assembled = VectorAssembler(
    inputCols=["Temperature","Humidity","Wind_Speed","General_Diffuse_Flows","Diffuse_Flows"],
    outputCol="weather",
    handleInvalid="skip"
)

In [35]:
pca = PCA(
    k=2,
    inputCol="weather",
    outputCol="pca_features"
)

In [36]:
label_maker = SQLTransformer(statement="""
    SELECT *, Power_Zone_3 AS label
    FROM __THIS__
    """
)

In [37]:
features_assembler = VectorAssembler(
    inputCols=["pca_features","day_night","Power_Zone_1","Power_Zone_2","Month_ohe"],
    outputCol="features",
    handleInvalid="skip"
)

In [38]:
lr = LinearRegression()

In [39]:
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

In [40]:
pipeline = Pipeline(stages = [
    cast_hour_double, 
    binary_day_night, 
    one_hot, 
    weather_assembled,
    pca,
    label_maker,
    features_assembler,
    lr
])

In [41]:
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(metricName='rmse'),
    numFolds=5,
    parallelism = 8
)

In [42]:
cvModel = cv.fit(spark_df)

In [43]:
best_pipeline_model = cvModel.bestModel
best_lr_model = best_pipeline_model.stages[-1] 

print("Best regParam:", best_lr_model._java_obj.getRegParam())
print("Best elasticNetParam:", best_lr_model._java_obj.getElasticNetParam())

Best regParam: 0.99
Best elasticNetParam: 0.75


In [46]:
rmse_values = cvModel.avgMetrics
best_rmse = min(rmse_values)

print("Best CV RMSE:", best_rmse)

Best CV RMSE: 2148.327320675894


In [50]:
train, test = spark_df.randomSplit([0.8,0.2], seed = 1)
cvModeltrain = cv.fit(train)

In [62]:
preds = cvModel.transform(test).select("label", "features", "prediction")

In [53]:
test_error = RegressionEvaluator().evaluate(cvModel.transform(test))
print(test_error)

2146.34268547728


In [65]:
preds_with_resids = preds.withColumn("residual",
    col("label") - col("prediction")
)

In [67]:
preds_with_resids.show(50)

+-----------+--------------------+------------------+-------------------+
|      label|            features|        prediction|           residual|
+-----------+--------------------+------------------+-------------------+
|14538.79518|(18,[0,1,3,4,6],[...|12571.791254030712| 1967.0039259692876|
|12826.98795|(18,[0,1,2,3,4,6]...|11582.519517257959| 1244.4684327420418|
|14353.73494|(18,[0,1,3,4,6],[...|12527.962405167902| 1825.7725348320982|
|12283.37349|(18,[0,1,2,3,4,6]...|10810.102935505765| 1473.2705544942346|
|14105.06024|(18,[0,1,3,4,6],[...|12390.810088535489| 1714.2501514645119|
| 15822.6506|(18,[0,1,3,4,6],[...|13623.559482434008|  2199.091117565993|
| 14897.3494|(18,[0,1,3,4,6],[...|12575.189189902037| 2322.1602100979617|
|15013.01205|(18,[0,1,3,4,6],[...| 12578.39449614699| 2434.6175538530097|
|15336.86747|(18,[0,1,3,4,6],[...|12994.584356664835| 2342.2831133351647|
|16794.21687|(18,[0,1,3,4,6],[...| 14936.30197025543|   1857.91489974457|
|18130.12048|(18,[0,1,3,4,6],[...|1651